# Function & Tool Calling — with LangSmith Tracing

**Case study:** mFinAgent — an NSE stock-analytics agent

This is the same DeepAgents + Groq tool-calling agent as `Tool_Calling.ipynb`, but with **[LangSmith](https://smith.langchain.com/) tracing** enabled. Because the agent is built on LangChain, every LLM call and every tool call is captured automatically once tracing is configured — no code changes to the agent itself.

Open the **GenAITraining** project in LangSmith to inspect, for each run:
- The full prompt sent to the LLM and its response
- Each tool the model decided to call, with inputs and outputs
- Token counts, latency, and the nested call tree of the agent loop

## Prerequisites

Install the required packages (adds `langsmith` on top of the originals):

```bash
pip install deepagents langchain langchain-groq yfinance pandas langsmith
```

You need two API keys:
- A **Groq** key — https://console.groq.com
- A **LangSmith** key — https://smith.langchain.com (Settings → API Keys)

In [ ]:
!pip install deepagents langchain langchain-groq yfinance pandas langsmith --quiet

## Enable LangSmith tracing

LangChain reads these environment variables on every call. Setting them **before** building the agent is all that's needed — tracing then happens transparently. We pin the project name to **GenAITraining** so all runs land in one place.

In [ ]:
import os
from getpass import getpass

# --- LangSmith tracing configuration ---
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_PROJECT"] = "GenAITraining"
os.environ["LANGSMITH_API_KEY"] = getpass("LANGSMITH_API_KEY:")

print(f"Tracing enabled -> project: {os.environ['LANGSMITH_PROJECT']}")

In [ ]:
# -- Imports --
import math
import json
import difflib
import numpy as np

import pandas as pd
import yfinance as yf

from langchain_core.tools import tool
from langchain_groq import ChatGroq
from deepagents import create_deep_agent

In [ ]:
os.environ["GROQ_API_KEY"] = getpass("GROQ_API_KEY:")

## The tool

`get_stock_returns` pulls real monthly data from yfinance and computes basic statistics. When the agent calls it, the tool's inputs and outputs show up as a child span under the run in LangSmith.

In [ ]:
@tool
def get_stock_returns(ticker: str, month: str):

    '''
        ticker: the stock ticker name
        month: number of month for which the stock trading data to used e.g. 1 or 2 or 3 etc.
        input string ticker is The correct ticker symbol for the stock in Nation Stock Exchange (NSE) India.
    '''

    # Get ticker name correctly
    msft = yf.Ticker(ticker.split(".")[0] + ".NS")

    # get historical market data
    hist = msft.history(period=month + "mo")

    # Compute the market data
    hist['daily_changes'] = (hist['Close'] - hist['Open']) * 100 / hist['Open']

    # Compute different statistics
    total_gain = (hist.iloc[-1]["Close"] - hist.iloc[0]["Open"]) * 100 / hist.iloc[0]["Open"]
    avg_daily_changes = np.mean(hist['daily_changes'])
    std_daily_changes = np.std(hist['daily_changes'])

    stock_stats = {'total_gain_in_percentage': round(total_gain, 3),
                   'average_daily_changes_in_percentage': round(avg_daily_changes, 3),
                   'std_daily_changes_in_percentage': round(std_daily_changes, 3)}

    return json.dumps(stock_stats)

In [ ]:
# Try it on HDFC Bank
result = get_stock_returns.invoke({"ticker": "HDFCBANK.NS", "month": "6"})
print(json.dumps(result, indent=2, default=str))

## Build the agent

Identical to the original notebook. Because tracing is already on, the agent and its internal tool-calling loop are captured automatically.

In [ ]:
SYSTEM = """You are mFinAgent, an Indian stock-analytics assistant.

When the user asks about a company:
1. Call `lookup_ticker` first to resolve the name to an NSE symbol.
   If multiple candidates score similarly, ask the user to disambiguate.
2. Call `fetch_monthly_returns` with the resolved ticker and time window.
3. If the user asks about risk, volatility, or probabilities, call
   `compute_statistics` on the returns list.
4. Always cite the actual numbers returned by the tools — never make them up.
"""

model = ChatGroq(model="openai/gpt-oss-120b", temperature=0.0)

agent = create_deep_agent(
    model=model,
    tools=[get_stock_returns],
    system_prompt=SYSTEM,
)

print("Agent ready.")

## Run queries (traced)

We pass a `run_name` and tags via the LangChain config so each run is easy to find in the **GenAITraining** project. Each `ask(...)` produces one trace whose tree shows the LLM deciding to call the tool, the tool execution, the result fed back, and the final answer.

In [ ]:
def ask(query):
    result = agent.invoke(
        {"messages": [{"role": "user", "content": query}]},
        config={"run_name": "mFinAgent-query", "tags": ["tool-calling", "mFinAgent"]},
    )

    print("=== FINAL ANSWER ===")
    print(result["messages"][-1].content)

In [ ]:
ask("What was return from HDFCBANK in last 6 months?")

In [ ]:
ask("between HDFCBANK and YESBANK, which stock has given better returns in last 3 months?")

## View the traces

Go to https://smith.langchain.com → project **GenAITraining**. Click any run to expand the tree and see:

- **LLM spans** — the exact messages sent (system prompt, conversation, tool schemas), the model's reply including any tool-call requests, and token/latency stats.
- **Tool spans** — `get_stock_returns` with the arguments the model chose and the JSON it returned.
- **The loop** — multiple LLM↔tool rounds nested under one agent run, so you can follow exactly how the agent reached its answer.